## optuna 설치

In [ ]:
!pip install optuna==4.6.0 optuna-integration==4.6.0

### Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 평가지표 함수 선언

In [ ]:
import sys
import os

sys.path.insert(0, '/content/drive/MyDrive/pill_detection_project')

from src.evaluation import (
    evaluate_all,
    init_history,
    update_history,
    save_history,
    load_history,
    plot_training_history,
    plot_compare_histories,
    convert_yolo_results,
    convert_torchvision_outputs,
)

## DataLoader

In [ ]:
import torch
import torchvision
import torch.optim as optim
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from src.preprocessing.dataset import get_loaders

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = '/content/drive/MyDrive/data/초급_프로젝트/dataset'
train_loader, val_loader, orig2model, num_classes, val_json = get_loaders(base_dir=BASE_DIR)

TEST_JSON_PATH = BASE_DIR + '/merged_annotations_test_final.json'

### 모델 정의

#### ✅ 변경사항
- **[개선 4]** Backbone을 `resnet50` → `mobilenet_v3_large` 로 교체
  - 학습 속도 단축 목적
  - 성능 저하가 크면 resnet50으로 롤백 가능

In [ ]:
############################################################
#  모델 정의 (Faster R-CNN + MobileNetV3 Large FPN)
#  [개선 4] Backbone: ResNet50 → MobileNetV3 Large (학습 속도 단축)
############################################################

def build_model_fasterrcnn_v2(num_classes):
    """
    Faster R-CNN + MobileNetV3 Large FPN 모델 정의 함수
    - [개선 4] ResNet50 → MobileNetV3 Large로 교체 (학습 속도 단축)
    - 사전학습된 가중치(COCO)를 사용하여 전이학습
    - 마지막 분류층을 우리 데이터셋 클래스 수에 맞게 교체
    """
    # MobileNetV3 Large FPN 기반 Faster R-CNN 로드
    model = torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn(
        weights="DEFAULT"
    )

    # 분류 head를 우리 데이터셋 클래스 개수에 맞게 교체
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    model.to(DEVICE)
    return model


############################################################
# 함수 호출
############################################################

model = build_model_fasterrcnn_v2(num_classes)

### 학습 루프

#### ✅ 변경사항 요약
| 항목 | 베이스라인 (v1) | 개선 (v2) |
|------|----------------|----------|
| Scheduler | StepLR (step=3, γ=0.1) | **CosineAnnealingLR** (부드러운 lr 감소) |
| Early Stopping | ❌ 없음 | **✅ patience=5 적용** |
| Gradient Clipping | ❌ 없음 | **✅ max_norm=5.0 적용** |
| Backbone | ResNet50 FPN | **MobileNetV3 Large FPN** |

In [ ]:
############################################################
#  학습 루프 v2
#  [개선 1] CosineAnnealingLR 적용
#  [개선 2] Early Stopping 적용 (patience=5)
#  [개선 3] Gradient Clipping 적용 (max_norm=5.0)
############################################################

def train_fasterrcnn_v2(model, train_loader, val_loader, num_epochs=10):
    optimizer = optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=1e-4
    )

    # [개선 1] StepLR → CosineAnnealingLR 교체
    # StepLR은 계단식으로 뚝뚝 lr을 줄이지만,
    # CosineAnnealingLR은 코사인 곡선처럼 부드럽게 줄여 학습 안정성 향상
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,  # 전체 epoch 동안 한 사이클
        eta_min=1e-6       # lr이 줄어드는 최솟값
    )

    # [개선 2] Early Stopping 설정
    patience = 5           # 몇 epoch 동안 val_loss 개선 없으면 멈출지
    best_val_loss = float('inf')
    early_stop_counter = 0
    best_model_state = None

    history = init_history()

    for epoch in range(1, num_epochs + 1):
        # -------------------------------------------------------
        # 1) Train
        # -------------------------------------------------------
        model.train()
        train_loss_sum = 0.0
        for images, targets in train_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()

            # [개선 3] Gradient Clipping 적용
            # loss가 갑자기 튀는 현상(exploding gradient) 방지
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            optimizer.step()
            train_loss_sum += losses.item()

        avg_train_loss = train_loss_sum / max(1, len(train_loader))

        # -------------------------------------------------------
        # 2) Validation loss
        # -------------------------------------------------------
        model.train()
        val_loss_sum = 0.0
        with torch.no_grad():
            for images, targets in val_loader:
                images = [img.to(DEVICE) for img in images]
                targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
                val_loss_sum += losses.item()

        avg_val_loss = val_loss_sum / max(1, len(val_loader))

        # -------------------------------------------------------
        # 3) 검증셋 예측
        # -------------------------------------------------------
        all_outputs = []
        all_image_ids = []
        model.eval()
        with torch.no_grad():
            for images, targets in val_loader:
                images = [img.to(DEVICE) for img in images]
                outputs = model(images)
                batch_image_ids = [t["image_id"].item() for t in targets]
                all_outputs.extend(outputs)
                all_image_ids.extend(batch_image_ids)

        val_predictions = convert_torchvision_outputs(all_outputs, all_image_ids)

        # -------------------------------------------------------
        # 4) 평가
        # -------------------------------------------------------
        metrics = evaluate_all(
            gt_json_path=val_json,
            predictions=val_predictions,
            conf_threshold=0.25,
            pr_iou_threshold=0.5,
            temp_json_path=f"faster_rcnn_v2_temp_eval_epoch_{epoch}.json",
            model2orig={v: k for k, v in orig2model.items()}
        )

        # -------------------------------------------------------
        # 5) 기록 및 출력
        # -------------------------------------------------------
        update_history(history, epoch=epoch, train_loss=avg_train_loss,
                       val_loss=avg_val_loss, metrics=metrics)

        current_lr = scheduler.get_last_lr()[0]
        print(f"[Epoch {epoch}/{num_epochs}] "
              f"train_loss: {avg_train_loss:.4f} | "
              f"val_loss: {avg_val_loss:.4f} | "
              f"lr: {current_lr:.6f}")

        scheduler.step()

        # -------------------------------------------------------
        # 6) [개선 2] Early Stopping 체크
        # -------------------------------------------------------
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            early_stop_counter = 0
            # 가장 좋은 모델 상태 저장
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            print(f"  ✅ val_loss 개선! best: {best_val_loss:.4f}")
        else:
            early_stop_counter += 1
            print(f"  ⚠️  val_loss 개선 없음 ({early_stop_counter}/{patience})")
            if early_stop_counter >= patience:
                print(f"\n🛑 Early Stopping! {patience} epoch 동안 개선 없어 학습 종료")
                break

    # 가장 성능 좋았던 모델 복원 후 저장
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print("\n✅ Best 모델 복원 완료")

    torch.save(model.state_dict(), "fasterrcnn_v2.pth")
    save_history(history, "history_faster_rcnn_v2.json")
    plot_training_history(history, title_prefix="Faster R-CNN v2")
    print("모델 저장 완료")
    return model


############################################################
# 함수 호출
############################################################

model = train_fasterrcnn_v2(model, train_loader, val_loader, num_epochs=10)

## 깃허브

In [ ]:
# 1. 최신 코드 받기
!cd /content/drive/MyDrive/pill_detection_project && git pull origin main

# 2. 새 브랜치 만들기
!cd /content/drive/MyDrive/pill_detection_project && git checkout -b feat/fasterrcnn-v2-improvements

# 3. 파일 추가 + 커밋 + push
!cd /content/drive/MyDrive/pill_detection_project && git add src/models/fasterrcnn/fasterrcnn_v2.ipynb && git commit -m "feat: Faster R-CNN v2 - CosineAnnealingLR, EarlyStopping, GradientClipping, MobileNetV3" && git push origin feat/fasterrcnn-v2-improvements

# 4. 깃허브 웹에서 PR 올리기
# https://github.com/wina0901/pill_detection_project 접속
# Compare & pull request 클릭 → Create pull request 클릭